# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
#loading the data from last week
!pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

base = "hf://datasets/FlyRank/internship-warehouse"

df_content = con.sql(f"SELECT * FROM read_parquet('{base}/dim_content.parquet')").df()
df_clients = con.sql(f"SELECT * FROM read_parquet('{base}/dim_clients.parquet')").df()
df_march = con.sql(f"SELECT * FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/data_0.parquet')").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
agg_df = df_march.groupby(['client_hash_id', 'content_hash_id']).agg(
    gsc_impressions=('gsc_impressions', 'sum'),
    gsc_clicks=('gsc_clicks', 'sum'),
    gsc_sum_position=('gsc_sum_position', 'sum'),
    ga4_sessions=('ga4_sessions', 'sum'),
    ga4_engaged_sessions=('ga4_engaged_sessions', 'sum'),
).reset_index()

agg_df['gsc_avg_position'] = agg_df['gsc_sum_position'] / agg_df['gsc_impressions']
agg_df.loc[agg_df['gsc_sum_position'] == 0, 'gsc_avg_position'] = pd.NA # treat gsc_avg_position == 0 as missing (no real position was ever recorded)

print(f"Pages with missing position (0 or NaN combined): {agg_df['gsc_avg_position'].isna().sum()}")

print(f"agg_df shape: {agg_df.shape}")
agg_df.head()

Pages with missing position (0 or NaN combined): 156133
agg_df shape: (331437, 8)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,ga4_sessions,ga4_engaged_sessions,gsc_avg_position
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,0,0,0,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,0,0,0,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,0,0,0,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9,0,0,9.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,0,0,0,NaN


## 1. My rule and its reason codes

**The rule:**

Following the session's 4-part flag structure (population, evidence, condition, action):

Population: pages with ≥50 GSC impressions OR ≥10 GA4 sessions (enough data to judge fairly; excludes pages too small to trust).

Evidence: CTR compared against the average CTR of other pages at a similar position (fair comparison, per Signal 1); engagement rate for pages with a reliable session count (per Signal 2).

Condition (checked in priority order): (1) CTR below half the position-peer average → snippet problem. (2) Otherwise, engagement rate below 10% with a reliable sample → content problem. (3) Otherwise → monitor.

Action & reason code: snippet_fix (CTR_below_half_position_peers), content_fix (engagement_below_10pct_reliable), or monitor (no_flag_triggered).

Score: estimated potential clicks gained if fixed (peer_avg_ctr − page_ctr × impressions) for snippet fixes; a smaller session-weighted priority score for content fixes. Mirrors the session's reasoning that Article A's fix was worth ~150 clicks/month versus Article D's ~12 — prioritize by real impact, not just by which flag fired.

**Signal 1: CTR vs. Position — CONFIRMED**

Checked whether pages ranking better on Google (lower average position) get a higher click-through-rate, as FlyRank's CTR-fix flag assumes.

| Position bucket | n (pages) | Avg CTR |
|---|---|---|
| 1-3 (top) | 18,860 | 1.17% |
| 4-10 | 83,288 | 0.49% |
| 11-20 | 29,922 | 0.33% |
| 21+ | 44,668 | 0.20% |

CTR drops consistently and monotonically as position worsens — confirming the assumption behind FlyRank's CTR-fix logic holds true in this data.

In [13]:
#Pages that actually got impressions
signal_df = agg_df[agg_df['gsc_impressions'] > 0].copy()

#CTR for each page
signal_df['ctr'] = signal_df['gsc_clicks'] / signal_df['gsc_impressions']

# Bucket pages by their average position
def position_bucket(pos):
    if pd.isna(pos):
        return 'no_position_data'
    elif pos <= 3:
        return '1-3 (top)'
    elif pos <= 10:
        return '4-10'
    elif pos <= 20:
        return '11-20'
    else:
        return '21+'

signal_df['position_bucket'] = signal_df['gsc_avg_position'].apply(position_bucket)

# Build the bucket table: average CTR per position bucket, with row counts (n)
ctr_position_table = signal_df.groupby('position_bucket').agg(
    n=('ctr', 'count'),
    avg_ctr=('ctr', 'mean')
).reindex(['1-3 (top)', '4-10', '11-20', '21+'])

print(ctr_position_table)

                     n   avg_ctr
position_bucket                 
1-3 (top)        17426  0.009961
4-10             83288  0.004873
11-20            29922  0.003285
21+              44668  0.001952


**Signal 2: Engagement Rate — CONFIRMED**

Checked whether pages with real, reliable traffic (≥10 sessions, to avoid tiny-sample noise) genuinely show a "shop problem" pattern — people arrive but don't engage — as described in this week's session.

Excluded 4,525 pages with fewer than 10 sessions as statistically unreliable (a single lucky/unlucky visitor can swing engagement rate to 0% or 100%).

| Engagement bucket | n (pages) | Avg sessions |
|---|---|---|
| high (60%+) | 0 | — |
| medium (30-59%) | 36 | 12.6 |
| low (10-29%) | 1,393 | 30.0 |
| very low (<10%) | 7,857 | 81.6 |

Among the 9,286 pages with a reliable sample size, 85% fall into the "very low" engagement bucket, and zero pages achieve high engagement. This confirms genuine content/engagement problems are widespread in this dataset — not just an artifact of small-sample noise. A content-fix flag has real, meaningful work to do here.

In [14]:
#signal 2: Engagement rate
#filtering to seessions with actual engagement
engagement_df = agg_df[agg_df['ga4_engaged_sessions'] > 0].copy()

engagement_df['engagement_rate'] = engagement_df['ga4_engaged_sessions'] / engagement_df['ga4_sessions'] # rate = engaged sessions / total sessions

def engagement_bucket(rate):
    if rate >= 0.6:
        return 'high (60%+)'
    elif rate >= 0.3:
        return 'medium (30-59%)'
    elif rate >= 0.1:
        return 'low (10-29%)'
    else:
        return 'very low (<10%)'

engagement_df['engagement_bucket'] = engagement_df['engagement_rate'].apply(engagement_bucket)

engagement_table = engagement_df.groupby('engagement_bucket').agg(
    n=('engagement_rate', 'count'),
    avg_sessions=('ga4_sessions', 'mean')
).reindex(['high (60%+)', 'medium (30-59%)', 'low (10-29%)', 'very low (<10%)'])

print(engagement_table)



                      n  avg_sessions
engagement_bucket                    
high (60%+)         784       1.21301
medium (30-59%)    1520      3.098026
low (10-29%)       3650     15.327123
very low (<10%)    7857     81.597047


It is seen that pages with a very high engagement rate has very low avg sessions number while it is opposite for pages with low engagement rates, these might be unreliable data and might lead to unfair comparisons

So we take a subset of our engagement_df with enough ga4 sessions to be meaningful

In [15]:
# engagement rate for pages with enough sessions to be meaningful
reliable_engagement = engagement_df[engagement_df['ga4_sessions'] >= 10].copy()

engagement_table_v2 = reliable_engagement.groupby('engagement_bucket').agg(
    n=('engagement_rate', 'count'),
    avg_sessions=('ga4_sessions', 'mean')
).reindex(['high (60%+)', 'medium (30-59%)', 'low (10-29%)', 'very low (<10%)'])

print(f"Pages with < 10 sessions (excluded as unreliable): {(engagement_df['ga4_sessions'] < 10).sum()}")
print(f"Pages with >= 10 sessions (reliable sample): {len(reliable_engagement)}")
print()
print(engagement_table_v2)

Pages with < 10 sessions (excluded as unreliable): 4525
Pages with >= 10 sessions (reliable sample): 9286

                      n  avg_sessions
engagement_bucket                    
high (60%+)        <NA>          <NA>
medium (30-59%)      36     12.611111
low (10-29%)       1393     30.016511
very low (<10%)    7857     81.597047


In [16]:
import pandas as pd

# Position bucket average CTR
position_avg_ctr = signal_df.groupby('position_bucket')['ctr'].mean().to_dict()

# a combined working table
rule_df = agg_df.copy()
rule_df['ctr'] = rule_df['gsc_clicks'] / rule_df['gsc_impressions']
rule_df['engagement_rate'] = rule_df['ga4_engaged_sessions'] / rule_df['ga4_sessions']
rule_df['position_bucket'] = rule_df['gsc_avg_position'].apply(position_bucket)
rule_df['peer_avg_ctr'] = rule_df['position_bucket'].map(position_avg_ctr)

# eligible pages only
eligible = (rule_df['gsc_impressions'] >= 50) | (rule_df['ga4_sessions'] >= 10)
rule_df = rule_df[eligible].copy()

def apply_rule(row):
    # snippet problem
    if row['gsc_impressions'] >= 50 and pd.notna(row['ctr']) and pd.notna(row['peer_avg_ctr']):
        if row['ctr'] < 0.5 * row['peer_avg_ctr']:
            gap = row['peer_avg_ctr'] - row['ctr']
            score = gap * row['gsc_impressions']  # potential extra clicks if fixed
            return pd.Series(['snippet_fix', 'CTR_below_half_position_peers', score])

    # content problem
    if row['ga4_sessions'] >= 10 and pd.notna(row['engagement_rate']):
        if row['engagement_rate'] < 0.10:
            score = row['ga4_sessions'] * (0.10 - row['engagement_rate']) * 10  # smaller priority weight
            return pd.Series(['content_fix', 'engagement_below_10pct_reliable', score])

    # monitor
    return pd.Series(['monitor', 'no_flag_triggered', 0])

rule_df[['action', 'reason_code', 'score']] = rule_df.apply(apply_rule, axis=1)

print(rule_df['action'].value_counts())
print(rule_df['reason_code'].value_counts())

action
snippet_fix    75663
monitor        28755
content_fix    12094
Name: count, dtype: int64
reason_code
CTR_below_half_position_peers      75663
no_flag_triggered                  28755
engagement_below_10pct_reliable    12094
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [17]:
import os

# Sort by score, descending (highest-impact opportunities first)
ranked_queue = rule_df.sort_values('score', ascending=False).reset_index(drop=True)

output_cols = ['client_hash_id', 'content_hash_id', 'action', 'reason_code', 'score',
               'gsc_impressions', 'gsc_clicks', 'ctr', 'gsc_avg_position',
               'ga4_sessions', 'engagement_rate']
ranked_queue_output = ranked_queue[output_cols]

# Make sure the output folder exists
os.makedirs('work/outputs', exist_ok=True)

ranked_queue_output.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Ranked queue written: {len(ranked_queue_output)} rows")
print(f"Top 5 highest-priority pages:")
ranked_queue_output.head(5)

Ranked queue written: 116512 rows
Top 5 highest-priority pages:


,client_hash_id,content_hash_id,action,reason_code,score,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,ga4_sessions,engagement_rate
0,client_23a62021009f63c4,content_36e53e9c707674fc,content_fix,engagement_below_10pct_reliable,2253.000000,194579,242,0.001244,32.786981,2603,0.013446
1,client_23a62021009f63c4,content_44f34c0a90047651,snippet_fix,CTR_below_half_position_peers,2091.787368,212404,24,0.000113,0.665877,37,0.027027
2,client_e547b89c05043229,content_8d7d99f109e19aa2,snippet_fix,CTR_below_half_position_peers,1738.063435,203497,289,0.001420,2.468557,164,0.097561
3,client_e547b89c05043229,content_0e03de7680314cd5,snippet_fix,CTR_below_half_position_peers,1484.501339,221310,720,0.003253,2.506100,465,0.126882
4,client_73cda7b4e4f265ea,content_8e1334d6356668e3,snippet_fix,CTR_below_half_position_peers,1343.595403,134984,1,0.000007,2.693038,4,0.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.